# Task A — transductive training on derivable cross-task labels

Task B is the hate subset of the same released corpus. The repository can therefore derive certain Task A labels for validation comments that recur in a Task B file: membership means `Hate`; a unique text match in Task A training can also supply a label. This notebook tests whether adding those **certain-only** labels to every training fold improves the current MuRIL recipe.

The control and transductive arms are matched five-fold runs on the original Task A training rows. The 365 derived validation rows are never used as OOF targets. Because they are used by the transductive arm, the released validation score is reported in two parts: the derived subset is a diagnostic and the non-derived subset is the honest external readout. Hidden Task A test rows are explicitly excluded. This notebook does not produce a submission ZIP.

In [ ]:
import os, pathlib, shutil, subprocess, sys
from pathlib import Path
WORK = '/kaggle/working/hastika'
if os.path.isdir(WORK + '/.git'):
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only', 'origin', 'task-b'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', 'task-b', '--depth', '1', 'https://github.com/robinpnalex/Hastika-ICON2026.git', WORK], check=True)
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))
pathlib.Path('artifacts/logs').mkdir(parents=True, exist_ok=True)
from urllib.request import urlretrieve
p = pathlib.Path('data/raw/hastika_binary_validation.csv')
if not p.exists():
    p.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve('https://raw.githubusercontent.com/robinpnalex/Hastika-ICON2026/task-b/data/raw/hastika_binary_validation.csv', p)
assert p.exists() and p.stat().st_size > 0
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, f1_score
from hastika.common.preprocessing import dedupe_index
from hastika.task_a.leak import derive
assert torch.cuda.is_available(), 'enable a GPU'
print('gpu:', torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print('$', ' '.join(map(str, cmd)), flush=True)
    fh = open(log, 'w') if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f'exit {p.returncode}: {cmd}')


## 1. Verify the derivable-label pool

In [ ]:
derived = derive(target='binary_validation_inputs.csv', include_uncertain=False, verbose=True)
assert len(derived) == 365 and derived['id'].is_unique
assert set(derived['Label'].unique()) <= {'Hate', 'Non-Hate'}
print(derived['Label'].value_counts().to_dict())
print(derived['route'].value_counts().to_dict())
print('Only these certain rows will be appended to each training fold; uncertain absent-everywhere rows are excluded.')


## 2. Run a matched control and transductive OOF fit

In [ ]:
CONTROL_TAG = 'a_run26_control'
TRANS_TAG = 'a_run26_transductive_certain'
common = [sys.executable, '-u', '-m', 'hastika.models.muril', '--model', 'google/muril-base-cased', '--folds', '5', '--seeds', '42', '--epochs', '6', '--bs', '8', '--grad-accum', '2', '--eval-bs', '32', '--select', 'last', '--reinit-layers', '2']
run(common + ['--tag', CONTROL_TAG], log=f'artifacts/logs/{CONTROL_TAG}.log')
run(common + ['--tag', TRANS_TAG, '--transductive', '--transductive-certain-only', '--no-hidden-test'], log=f'artifacts/logs/{TRANS_TAG}.log')
print('matched control and transductive OOF fits completed')


## 3. Report OOF and released-validation diagnostics

In [ ]:
train = pd.read_csv('data/raw/binary_train.csv')
keep = dedupe_index(train['Comment'].tolist(), train['Label'].tolist(), 'task A')
fit_df = train.iloc[keep].reset_index(drop=True)
y = (fit_df['Label'] == 'Hate').astype(int).to_numpy()
test_ids = pd.read_csv('data/raw/binary_validation_inputs.csv')[['id']]
val = test_ids.merge(pd.read_csv('data/raw/hastika_binary_validation.csv'), on='id', how='left', validate='one_to_one')
assert val['Label'].notna().all() and val['id'].is_unique
derived_ids = set(derived['id'])
mask_derived = val['id'].isin(derived_ids).to_numpy()
rows = []
for tag in [CONTROL_TAG, TRANS_TAG]:
    d = Path('artifacts/runs') / tag
    oof = np.load(d / 'oof_probs.npy')
    test = np.load(d / 'test_probs.npy')
    pred = oof.argmax(1)
    rows.append({'arm': tag, 'split': 'OOF original train', 'rows': len(y), 'macro_f1': f1_score(y, pred, average='macro'), 'accuracy': accuracy_score(y, pred)})
    p = test.argmax(1)
    target = (val['Label'] == 'Hate').astype(int).to_numpy()
    for name, m in [('released validation overall', np.ones(len(val), dtype=bool)), ('derived subset (diagnostic)', mask_derived), ('non-derived subset (honest)', ~mask_derived)]:
        rows.append({'arm': tag, 'split': name, 'rows': int(m.sum()), 'macro_f1': f1_score(target[m], p[m], average='macro'), 'accuracy': accuracy_score(target[m], p[m])})
summary = pd.DataFrame(rows)
OUT = Path('/kaggle/working/task_a_run26_outputs')
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / 'summary.csv', index=False)
print(summary.to_string(index=False))
print('Interpret the non-derived validation row as the primary external comparison; the derived row was used for transductive training.')
print('DOWNLOAD RESULTS FROM:', OUT, 'and artifacts/runs')


## 4. Disclosure and reproducibility note

The transductive arm uses 365 labels derived from released Task B overlap, of which 359 are Hate and 6 are Non-Hate. It excludes uncertain labels and hidden-test rows. Any downstream submission using this arm must disclose that cross-task derivation and must not be compared directly with a non-transductive score without stating the difference.